In [ ]:
from pathlib import Path

import polars as pl
import polars.selectors as cs
import matplotlib.pyplot as plt

from climate_attitudes.settings import Config
from climate_attitudes.dataset import Dataset
from climate_attitudes import configure_mpl
from climate_attitudes.visualisation import plot_corr_with_dendro
import climate_attitudes.datasets.imputed as ds_spec
from climate_attitudes.correlation import Correlation

FONT_PATH = Path("../fonts")
configure_mpl(FONT_PATH)

plt.rc("figure", dpi=150)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(config, name="ds1")
# dataset_std = dataset.standardise(cs.exclude("participant_id", "wave"))
resp = dataset.response.collect()  # .sample(n=1000)
# resp_std = dataset_std.response.collect()

In [ ]:
plot_corr_with_dendro(
    resp.with_columns(cs.exclude(*ds_spec.SURVEY_COLS).cast(pl.Float64)).drop("cc1"),
    kind=Correlation.PEARSON,
    no_cbar=True,
    x_categories=[c for c in ds_spec.CATEGORIES if c != "cc1"],
    y_categories=[c for c in ds_spec.CATEGORIES if c != "cc1"],
    y_vars=["cc3"],
    row_cluster=False,
)

In [ ]:
plot_corr_with_dendro(
    resp.with_columns(cs.exclude(*ds_spec.SURVEY_COLS).cast(pl.Float64))
    .filter(cc1=2)
    .drop("cc1"),
    kind=Correlation.PEARSON,
    no_cbar=True,
    x_categories=[c for c in ds_spec.CATEGORIES if c != "cc1"],
    y_categories=[c for c in ds_spec.CATEGORIES if c != "cc1"],
)

In [ ]:
plot_corr_with_dendro(
    resp.with_columns(cs.exclude(*ds_spec.SURVEY_COLS).cast(pl.Float64)),
    kind=Correlation.PARTIAL,
    no_cbar=True,
    x_categories=ds_spec.CATEGORIES,
    y_categories=ds_spec.CATEGORIES,
)

In [ ]:
(
    resp.select("pol8", "pol8_pi")
    .select(
        pl.all().replace_strict({-1: "Liberal", -2: "Conservative"}, return_dtype=str)
    )
    .group_by("pol8", "pol8_pi")
    .agg(pl.len().alias("count"))
    .with_columns(prop=(pl.col("count") / pl.col("count").sum()))
    .drop("count")
    .sort(by=("pol8", "pol8_pi"))
)

In [ ]:
(
    resp.filter(pl.col("wave").is_in([3, 4]))
    .select("pol8", "pol8_pi")
    .select(
        pl.all().replace_strict({-1: "Liberal", -2: "Conservative"}, return_dtype=str)
    )
    .group_by("pol8", "pol8_pi")
    .agg(pl.len().alias("count"))
    .with_columns(prop=(pl.col("count") / pl.col("count").sum()))
    .drop("count")
    .sort(by=("pol8", "pol8_pi"))
)